# Session 3: floating is not enough, it must also right itself
**A. Real-world problem.** A barge can float with plenty of freeboard and still roll over
when the cargo is stacked too high. Stability asks a different question from buoyancy:
when the barge is pushed over by a small angle, does the water push it back? The answer
is the metacentric height GM. We compute it by formula, then check it against an exact
numerical calculation of the heeled cross-section.
*Educational 2D model only: not a naval architecture tool; it ignores waves, wind,
flooding, free liquid surfaces and the rules that govern real ships.*

In [ ]:
import pathlib

import numpy as np

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
RHO = 1000.0            # fresh water, kg/m^3
LEN, BEAM, DEPTH = 20.0, 6.0, 2.5      # barge length, beam and hull depth, m
LIGHT, KG_LIGHT = 60000.0, 1.0         # empty barge: mass kg, centre of gravity above keel m

## Step 1: the small-angle formulas
Heights are measured up from the keel (the bottom): K is the keel, B the centre of
buoyancy (centre of the displaced water), G the centre of gravity, M the metacentre.
For a box floating at draft d:  KB = d / 2,  BM = I / V with I = L B^3 / 12 (waterplane)
and V = L B d, so BM = B^2 / (12 d). Then KM = KB + BM and  GM = KM - KG.
**GM > 0: the box rights itself. GM < 0: it heels over.**

In [ ]:
def draft_for(mass, beam=BEAM, length=LEN, rho=RHO):
    return mass / (rho * length * beam)


def kb_box(draft):
    return draft / 2.0


def bm_box(draft, beam=BEAM):
    return beam ** 2 / (12.0 * draft)


def kg_loaded(cargo, cargo_h, light=LIGHT, kg_light=KG_LIGHT):
    """Centre of gravity (m above keel) of barge plus cargo (kg) whose own centre is at height cargo_h."""
    return (light * kg_light + cargo * cargo_h) / (light + cargo)


def gm_loaded(cargo, cargo_h, beam=BEAM, length=LEN, light=LIGHT, kg_light=KG_LIGHT):
    d = draft_for(light + cargo, beam, length)
    return kb_box(d) + bm_box(d, beam) - kg_loaded(cargo, cargo_h, light, kg_light)

## Step 2: exact geometry when the box is heeled
Rotate the rectangle by the heel angle phi, find the horizontal water line that leaves the
same submerged area (same weight of water displaced), and take the centroid of the wet part:
that is the heeled centre of buoyancy B. The righting arm GZ is the horizontal distance
between the upward buoyant force (through B) and the downward weight (through G):
**GZ = y_G - y_B** with y measured sideways in the water frame. Righting moment = weight * GZ.

In [ ]:
def clip_below(poly, level):
    """Part of a polygon (list of (y, z) points) with z <= level (Sutherland-Hodgman)."""
    out = []
    n = len(poly)
    for i in range(n):
        p, q = poly[i], poly[(i + 1) % n]
        pin, qin = p[1] <= level, q[1] <= level
        if pin:
            out.append(p)
        if pin != qin:
            t = (level - p[1]) / (q[1] - p[1])
            out.append((p[0] + t * (q[0] - p[0]), level))
    return out


def area_centroid(poly):
    if len(poly) < 3:
        return 0.0, 0.0, 0.0
    y = np.array([p[0] for p in poly])
    z = np.array([p[1] for p in poly])
    y2, z2 = np.roll(y, -1), np.roll(z, -1)
    cr = y * z2 - y2 * z
    a = cr.sum() / 2.0
    if abs(a) < 1e-14:
        return 0.0, 0.0, 0.0
    return a, ((y + y2) * cr).sum() / (6 * a), ((z + z2) * cr).sum() / (6 * a)


def wet_section(phi, area, beam=BEAM, depth=DEPTH):
    """Heel angle phi (radians), wet cross-section area (m^2) -> (waterline height, y_B, z_B).
    Coordinates are in the water frame: y sideways, z up, origin where the middle of the keel
    would be if the box were upright and then turned about it."""
    if area >= beam * depth:
        raise ValueError("the box would be fully submerged")
    c, s = np.cos(phi), np.sin(phi)
    corners = [(-beam / 2, 0.0), (beam / 2, 0.0), (beam / 2, depth), (-beam / 2, depth)]
    world = [(y * c - z * s, y * s + z * c) for y, z in corners]
    lo = min(p[1] for p in world)
    hi = max(p[1] for p in world)
    for _ in range(80):
        mid = 0.5 * (lo + hi)
        a = area_centroid(clip_below(world, mid))[0]
        if a < area:
            lo = mid
        else:
            hi = mid
    level = 0.5 * (lo + hi)
    a, yb, zb = area_centroid(clip_below(world, level))
    return level, yb, zb


def righting_arm(phi, mass, kg, beam=BEAM, length=LEN, depth=DEPTH, rho=RHO, shift=0.0):
    """GZ (m) at heel angle phi (radians). `shift` moves G sideways in the hull by that many metres (cargo shift)."""
    area = mass / (rho * length)
    level, yb, zb = wet_section(phi, area, beam, depth)
    yg = -kg * np.sin(phi) + shift * np.cos(phi)
    return yg - yb


def gz_curve(phis, mass, kg, **kw):
    return np.array([righting_arm(p, mass, kg, **kw) for p in phis])


def wall_sided_gz(phi, gm, bm):
    """Analytic GZ for a box while the deck edge is dry and the bilge stays wet."""
    return np.sin(phi) * (gm + 0.5 * bm * np.tan(phi) ** 2)

## Step 3: cargo shift and a failure case
A cargo moved sideways by s moves G sideways by gs = cargo * s / mass. The barge then heels
to the angle where the righting arm equals the offset, GZ(phi) = gs * cos(phi). With a
negative GM the upright position is unstable: the barge settles at an "angle of loll"
where GZ is zero again (or capsizes).

In [ ]:
def heel_after_shift(cargo, cargo_h, s, light=LIGHT, **kw):
    """Equilibrium heel angle (radians) after moving the cargo sideways by s metres (scan, then bisection on the exact GZ)."""
    mass = light + cargo
    kg = kg_loaded(cargo, cargo_h, light)
    gs = cargo * s / mass
    f = lambda p: righting_arm(p, mass, kg, **kw) - gs * np.cos(p)
    grid = np.linspace(0.0, np.radians(80.0), 161)
    vals = [f(p) for p in grid]
    if gs == 0 or vals[0] > 0:
        return 0.0
    for a, b, fa, fb in zip(grid[:-1], grid[1:], vals[:-1], vals[1:]):
        if fa <= 0 < fb:
            lo, hi = a, b
            break
    else:
        return float("nan")      # no equilibrium: it capsizes
    for _ in range(60):
        mid = 0.5 * (lo + hi)
        if f(mid) <= 0:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def angle_of_loll(mass, kg, **kw):
    """Smallest positive heel angle (radians) where GZ returns to zero (for GM < 0), by exact geometry."""
    grid = np.linspace(np.radians(0.5), np.radians(60.0), 120)
    vals = [righting_arm(p, mass, kg, **kw) for p in grid]
    for a, b, fa, fb in zip(grid[:-1], grid[1:], vals[:-1], vals[1:]):
        if fa < 0 <= fb:
            lo, hi = a, b
            for _ in range(60):
                mid = 0.5 * (lo + hi)
                if righting_arm(mid, mass, kg, **kw) < 0:
                    lo = mid
                else:
                    hi = mid
            return 0.5 * (lo + hi)
    return float("nan")


def max_cargo_height(cargo, gm_min=0.15):
    """Highest cargo centre of gravity (m above keel) that keeps GM >= gm_min (closed form from GM = KM - KG)."""
    mass = LIGHT + cargo
    d = draft_for(mass)
    km = kb_box(d) + bm_box(d)
    kg_max = km - gm_min
    return (kg_max * mass - LIGHT * KG_LIGHT) / cargo

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    cargo, h = 120000.0, 2.5
    mass = LIGHT + cargo
    d = draft_for(mass)
    kg = kg_loaded(cargo, h)
    gm = gm_loaded(cargo, h)
    bm = bm_box(d)
    print(f"draft {d:.3f} m  KB {kb_box(d):.3f}  BM {bm:.3f}  KM {kb_box(d) + bm:.3f}  KG {kg:.3f}  GM {gm:.3f} m")
    phis = np.radians(np.arange(0, 81, 1.0))
    gz = gz_curve(phis, mass, kg)
    for deg in (1, 5, 10, 20):
        p = np.radians(deg)
        print(f"heel {deg:2d} deg: exact GZ {righting_arm(p, mass, kg):.4f}  GM sin(phi) {gm * np.sin(p):.4f}  wall-sided {wall_sided_gz(p, gm, bm):.4f}")
    p1 = np.radians(0.5)
    print(f"GZ / sin(phi) at 0.5 deg: {righting_arm(p1, mass, kg) / np.sin(p1):.4f}  (GM = {gm:.4f})")
    print(f"deck edge goes under at {np.degrees(np.arctan(2 * (DEPTH - d) / BEAM)):.1f} deg, bilge lifts at {np.degrees(np.arctan(2 * d / BEAM)):.1f} deg")

    for hc in (2.0, 2.5, 3.0, 3.4, 3.8):
        print(f"cargo centre at {hc:.1f} m: KG {kg_loaded(cargo, hc):.3f}, GM {gm_loaded(cargo, hc):+.3f}")
    kg_bad = kg_loaded(cargo, 3.8)
    loll = angle_of_loll(mass, kg_bad)
    print(f"failure case h=3.8 m: GM {gm_loaded(cargo, 3.8):+.3f} m, angle of loll {np.degrees(loll):.1f} deg"
          f" (analytic tan^2 = -2 GM / BM -> {np.degrees(np.arctan(np.sqrt(-2 * gm_loaded(cargo, 3.8) / bm))):.1f} deg)")
    print(f"max cargo height for GM >= 0.15 m with 120 t: {max_cargo_height(cargo):.2f} m")
    for s in (0.1, 0.2, 0.3):
        print(f"cargo shifted {s} m: heel {np.degrees(heel_after_shift(cargo, h, s)):.1f} deg  (small-angle estimate {np.degrees(np.arctan(cargo * s / mass / gm)):.1f} deg)")

    # figure 1: GZ curve versus small-angle line
    fig, ax = plt.subplots(figsize=(6.5, 4.2))
    ax.plot(np.degrees(phis), gz, label="exact geometry (loaded, cargo at 2.5 m)")
    ax.plot(np.degrees(phis), gm * np.sin(phis), "--", label="small angle: GM sin(phi)")
    ax.plot(np.degrees(phis), wall_sided_gz(phis, gm, bm), ":", label="wall-sided formula")
    gz_bad = gz_curve(phis, mass, kg_bad)
    ax.plot(np.degrees(phis), gz_bad, label="cargo at 3.8 m (GM negative)")
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_ylim(-0.4, 1.0)
    ax.set_xlabel("heel angle (degrees)")
    ax.set_ylabel("righting arm GZ (m)")
    ax.set_title("Righting arm of the loaded barge")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "gz_curve.png", dpi=120)

    # figure 2: section pictures at 20 deg for stable and unstable
    fig2, axs = plt.subplots(1, 2, figsize=(9, 3.8))
    for axp, hc, title in zip(axs, (2.5, 3.8), ("cargo low (2.5 m): GM > 0", "cargo high (3.8 m): GM < 0")):
        kgx = kg_loaded(cargo, hc)
        phi = np.radians(20)
        level, yb, zb = wet_section(phi, mass / (RHO * LEN))
        c, s = np.cos(phi), np.sin(phi)
        corners = np.array([(-BEAM / 2, 0), (BEAM / 2, 0), (BEAM / 2, DEPTH), (-BEAM / 2, DEPTH), (-BEAM / 2, 0)])
        wy = corners[:, 0] * c - corners[:, 1] * s
        wz = corners[:, 0] * s + corners[:, 1] * c - level
        axp.plot(wy, wz, "k-")
        axp.axhline(0, color="tab:blue")
        axp.plot(-kgx * s, kgx * c - level, "ro", label="G")
        axp.plot(yb, zb - level, "go", label="B")
        km = kb_box(d) + bm
        axp.plot(-km * s, km * c - level, "b^", label="M (initial)")
        axp.set_title(title)
        axp.set_aspect("equal")
        axp.set_xlabel("sideways position (m)")
        axp.legend(fontsize=8, loc="upper left")
    axs[0].set_ylabel("height above the water line (m)")
    fig2.tight_layout()
    fig2.savefig(OUT / "heeled_sections.png", dpi=120)

    # figure 3: GM versus cargo height
    hh = np.linspace(1.0, 5.0, 81)
    fig3, ax3 = plt.subplots(figsize=(6, 4))
    ax3.plot(hh, [gm_loaded(cargo, x) for x in hh])
    ax3.axhline(0, color="red", ls="--", label="GM = 0 (neutral)")
    ax3.axhline(0.15, color="orange", ls=":", label="GM = 0.15 m (design margin)")
    ax3.set_xlabel("height of the cargo's centre of gravity above the keel (m)")
    ax3.set_ylabel("metacentric height GM (m)")
    ax3.set_title("Loading high eats the stability (120 t cargo)")
    ax3.grid(alpha=0.3)
    ax3.legend(fontsize=8)
    fig3.tight_layout()
    fig3.savefig(OUT / "gm_vs_height.png", dpi=120)

## Where this stops
Real ships have curved hulls, free-surface liquids, waves and wind, and are judged by rules
that look at the whole GZ curve. See `promo/roadmap.md` for the road to the Target.

## Coding tasks
1. **Easy.** Print GM for cargo heights 2.0, 3.0 and 3.8 m with 120 t. Predict the signs first, then check them against the figure.
2. **Stretch.** Compute the GZ curve for 160 t at 2.5 m and find the heel angle where the deck edge goes under and where GZ is largest.
3. **Explain.** Explain why moving cargo sideways by 0.2 m heels the barge, and why the same shift heels it less when GM is larger.

## Assessment
- **A. Conceptual.** A barge has 1.2 m of freeboard but capsizes when loaded with tall stacked cargo. Explain why freeboard does not guarantee stability, and what quantity does.
- **B. Numerical.** The same barge carries 120 t with its centre of gravity at 3.0 m. Find KG and GM, and say whether it is stable.
- **C. Graph and simulation interpretation.** From the GZ figure, at 20 degrees the exact curve is above the small-angle line. Read approximately both values for the loaded barge, state which is larger and give a likely reason for the difference.
- **D. Design and optimisation challenge.** Design: with 160 t of cargo (instead of 120 t), find the highest cargo centre of gravity, in steps of 0.1 m, that keeps GM at or above 0.15 m. Compare it with the 3.40 m for 120 t and explain.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    cargo, h = 120000.0, 2.5
    mass = L.LIGHT + cargo
    d = L.draft_for(mass)
    kg = L.kg_loaded(cargo, h)
    gm = L.gm_loaded(cargo, h)
    bm = L.bm_box(d)
    checks.append((f"worked example: draft 1.500 m, KB 0.750, BM 2.000 (got {d:.3f}, {L.kb_box(d):.3f}, {bm:.3f})",
                   abs(d - 1.5) < 1e-12 and abs(L.kb_box(d) - 0.75) < 1e-12 and abs(bm - 2.0) < 1e-12))
    checks.append((f"worked example: KG = 2.000 m and GM = 0.750 m (got {kg:.3f}, {gm:.3f})", abs(kg - 2.0) < 1e-12 and abs(gm - 0.75) < 1e-12))

    # exact geometry in the small-angle limit equals GM sin(phi)
    p = np.radians(0.5)
    ratio = L.righting_arm(p, mass, kg) / np.sin(p)
    checks.append((f"validation: GZ / sin(phi) tends to GM at small angle (got {ratio:.4f})", abs(ratio - gm) < 2e-3))
    p10 = np.radians(10)
    checks.append(("validation: exact GZ matches the wall-sided formula at 10 degrees",
                   abs(L.righting_arm(p10, mass, kg) - L.wall_sided_gz(p10, gm, bm)) < 1e-6))
    checks.append(("validation: small-angle GM sin(phi) is within 5 percent at 10 degrees",
                   abs(L.righting_arm(p10, mass, kg) - gm * np.sin(p10)) / L.righting_arm(p10, mass, kg) < 0.05))
    checks.append(("exact geometry: zero heel gives zero righting arm", abs(L.righting_arm(0.0, mass, kg)) < 1e-9))
    checks.append(("exact geometry: righting arm is odd in the heel angle",
                   abs(L.righting_arm(-p10, mass, kg) + L.righting_arm(p10, mass, kg)) < 1e-9))
    level, yb, zb = L.wet_section(0.0, mass / (L.RHO * L.LEN))
    checks.append(("upright: waterline at the draft and B at half the draft", abs(level - d) < 1e-9 and abs(zb - d / 2) < 1e-9 and abs(yb) < 1e-9))
    a_wet = mass / (L.RHO * L.LEN)
    lv, _, _ = L.wet_section(np.radians(15), a_wet)
    checks.append(("heeled water line leaves the same wet area (weight is carried)",
                   abs(L.area_centroid(L.clip_below([(y * np.cos(np.radians(15)) - z * np.sin(np.radians(15)),
                                                        y * np.sin(np.radians(15)) + z * np.cos(np.radians(15)))
                                                       for y, z in [(-3, 0), (3, 0), (3, 2.5), (-3, 2.5)]], lv))[0] - a_wet) < 1e-6))

    # loading height
    checks.append(("raising the cargo lowers GM", L.gm_loaded(cargo, 3.0) < L.gm_loaded(cargo, 2.0)))
    hmax = L.max_cargo_height(cargo)
    checks.append((f"hand calculation: highest cargo centre for GM = 0.15 m is 3.40 m (got {hmax:.3f})", abs(hmax - 3.4) < 1e-9))
    checks.append(("GM is above 0.15 m at 3.39 m and negative at 3.7 m",
                   L.gm_loaded(cargo, 3.39) > 0.15 and L.gm_loaded(cargo, 3.7) < 0))

    # failure case
    gm_bad = L.gm_loaded(cargo, 3.8)
    kg_bad = L.kg_loaded(cargo, 3.8)
    checks.append((f"failure case: cargo at 3.8 m gives negative GM (got {gm_bad:.3f})", gm_bad < 0))
    checks.append(("failure case: a small heel makes the arm point the wrong way (GZ < 0)", L.righting_arm(np.radians(3), mass, kg_bad) < 0))
    loll = np.degrees(L.angle_of_loll(mass, kg_bad))
    analytic = np.degrees(np.arctan(np.sqrt(-2 * gm_bad / bm)))
    checks.append((f"failure case: angle of loll {loll:.1f} deg is near the analytic {analytic:.1f} deg", abs(loll - analytic) < 0.5))

    # cargo shift
    heel = np.degrees(L.heel_after_shift(cargo, h, 0.1))
    est = np.degrees(np.arctan(cargo * 0.1 / mass / gm))
    checks.append((f"cargo shift of 0.1 m: heel {heel:.1f} deg close to the small-angle estimate {est:.1f} deg", abs(heel - est) < 0.3))
    checks.append(("no shift gives no heel", L.heel_after_shift(cargo, h, 0.0) == 0.0))
    checks.append(("a bigger shift gives a bigger heel", L.heel_after_shift(cargo, h, 0.2) > L.heel_after_shift(cargo, h, 0.1)))

    # mini-project (the lever really matters): both limits, the heel limit governs
    best_gm = max(k * 0.05 for k in range(20, 100) if L.gm_loaded(cargo, k * 0.05) >= 0.15 - 1e-9)
    best_heel = max(k * 0.05 for k in range(20, 100)
                    if L.gm_loaded(cargo, k * 0.05) >= 0.15 - 1e-9 and np.degrees(L.heel_after_shift(cargo, k * 0.05, 0.2)) <= 10.0)
    checks.append((f"mini-project: GM limit allows cargo up to {best_gm:.2f} m, heel limit only up to {best_heel:.2f} m",
                   abs(best_gm - 3.4) < 1e-9 and abs(best_heel - 2.5) < 1e-9))
    return checks

import types
_ns = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
_ok = True
for _label, _passed in run_checks(_ns):
    print(('PASS  ' if _passed else 'FAIL  ') + _label)
    _ok &= bool(_passed)
print('PASS' if _ok else 'FAIL')
